# 7.6 优化步骤③：核内流水（RegBase / VF 编程）

## 1. MemBase vs RegBase

普通 Ascend C（MemBase 编程）的数据流：`GM → UB → GM`，Vector 计算的每条指令都要从 UB 读操作数、写回 UB。**RegBase 编程**加了一级：

```
MemBase： GM → UB → GM
RegBase： GM → UB → RegisterFile → UB → GM
```

数据装进向量寄存器后，多条计算指令可以**在寄存器间连续流转**，不必每条都回 UB——这是核内最后一级流水。

## 2. RegBase 的寄存器抽象

| 寄存器 | 数据类型 | 职责 |
|--|--|--|
| 向量寄存器 | RegTensor | 存放计算指令的目标数据（VF 计算的操作数与结果） |
| 地址寄存器 | AddrReg | 跟随 for-loop 迭代累加，更新 UB 上搬运数据的起始地址 |
| 对齐寄存器 | UnalignReg | 访问 UB 上非 32B 对齐的地址 |
| 掩码寄存器 | MaskReg | 按 bit 标识 RegTensor 中的有效数据（0 为无效），控制指令只作用于有效位 |

配套硬件单元：**寄存器读单元**（LD 指令：UB → 向量寄存器）、**寄存器写单元**（ST 指令：向量寄存器 → UB）、**寄存器计算单元**（对 RegTensor 计算源与目的都是 RegTensor）、**VF-Scalar 单元**（VF 函数内的标量计算）。

## 3. vLoop：RegBase 编程的核心

寄存器大小 256B，**所有循环按 256B 切分设计**。以一个 Mul + 缩放函数为例（完整保留了工程代码的结构）：

```cpp
template <typename T>
__aicore__ inline void MulQr_VF(const LocalTensor<T>& out, const LocalTensor<T>& in,
                                const LocalTensor<T>& scaleLocal, float quantScale,
                                uint64_t size, uint64_t align) {
    constexpr uint16_t floatRepSize = 64;               // 一个寄存器存 64 个 FP32
    uint16_t repeatTimes = (size + 64 - 1) / 64;        // 循环次数（尾块向上取整）
    __VEC_SCOPE__ {
        __ubuf__ T* inBuf  = (__ubuf__ T*)in.GetPhyAddr();
        __ubuf__ T* outBuf = (__ubuf__ T*)out.GetPhyAddr();
        MicroAPI::MaskReg pregAll = MicroAPI::CreateMask<T, MaskPattern::ALL>();
        for (uint16_t i = 0; i < repeatTimes; i++) {
            MicroAPI::RegTensor<T> vregSrc, vregScale, vregRes;
            uint16_t offset = i * floatRepSize;         // 数据偏移
            MicroAPI::DataCopy<T, LoadDist::DIST_NORM>(vregSrc, inBuf + offset);      // LD
            MicroAPI::DataCopy<T, LoadDist::DIST_BRC_B32>(vregScale, scaleBuf + ...);  // 广播 LD
            MicroAPI::Mul<T, MaskMergeMode::ZEROING>(vregRes, vregSrc, vregScale, pregAll);
            MicroAPI::DataCopy<T, StoreDist::DIST_NORM>(outBuf + offset, vregRes, pregAll); // ST
        }
    }
}
```

要点：

- **repeatTimes = ceil(size / 64)**：vLoop 次数由数据量与寄存器宽度直接决定，尾块用 MaskReg 兜底；
- **LD / 计算 / ST 三类微指令**：`DataCopy`（LD/ST，含 DIST_NORM 普通分布 / DIST_BRC_B32 广播分布）、`Duplicate`、`Div`、`Mul` 等；
- **广播加载**（DIST_BRC_B32）把每行的 scale 直接展开到寄存器，行乘无需循环；
- 常用 API 覆盖几乎所有基础 Vector 操作，写代码时以 IDP 文档为准。

## 4. VF 编写的五个要点

1. **评估改动范围**：VF 设计的是 Vector 相关计算，代码是「GM 搬到 UB 后计算的逻辑」——先圈定模块改动面；
2. **方案先行**：VF 很难调试，动笔前对每一步运算深入学习、确认运算逻辑；
3. **UB 数据形态**：数据在 UB 上以什么形式呈现（ND/NZ/分块），直接决定 vLoop 怎么设计；
4. **256B 切分**：一切循环围绕寄存器宽度设计，把传入形状合理切分；
5. **多查文档**：基础操作基本都有现成 API，不要手搓。

## 5. VF 的问题定位思路

VF 代码无法像普通代码那样打断点：

1. **先排除外围**：UB 上的值都能用调测指令打印——先确认错误发生在 VF 代码内部还是外部；
2. **手推公式**：保证数据搬运没问题后，从 vLoop 开始逐条手推计算，验证哪一步出错；
3. **搬出验证**：把寄存器数据搬回 UB 再打印，对比预期。

> Softmax VF（7.5 节的函数族）就是这条路数的集大成者——第 8 章将展开 VF 的完整优化方法论（VF 建模、DN 等）。

## 小测验

1. MemBase 与 RegBase 的数据流差在哪一级？这一级带来什么收益？
2. 四类寄存器各自的职责是什么？LD/ST 指令由哪些单元处理？
3. vLoop 的循环次数怎么算？尾块靠什么兜底？
4. VF 难调试，给出完整的定位三步法。